# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Current executable contract (starter slice):** one row = one pseudonymized content item at one current decision snapshot. The bundled starter contains trailing-90-day search/engagement measurements for **30,000 content items across 32 pseudonymized clients**. It does not expose a daily report date, so it is appropriate for validating the task mechanics but not for defining a true future outcome.

**Capstone warehouse contract:** one row = one `content_hash_id` at one decision date. Features will be built strictly from a historical window ending at the decision date; the eventual target must be observed strictly after that point. The gated warehouse spans **2025-01-27 through 2026-06-30**, with the daily fact table at `report_date × client_hash_id × content_hash_id` grain. Development queries should use a mid-panel partition such as `month=2026-03`, while the final month remains sealed for later evaluation.

The notebook contains the exact DuckDB query block for the warehouse. It runs automatically only when `HF_TOKEN` is present in the environment; the token is never stored in the notebook.

In [1]:
import os, json, subprocess
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root = find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root = Path(REPO_DIR).resolve()

os.chdir(root)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Starter rows:", len(df))
print("Starter clients:", df["client_id"].nunique())
print("One row per content item:", df["content_id"].is_unique)
print("Starter fields:", df.shape[1])


Starter rows: 30000
Starter clients: 32
One row per content item: True
Starter fields: 44


## 2. Fields: feature / label / context / excluded

### Features — knowable before the review decision
- `impressions_90d`, `clicks_90d`, `sessions_90d`
- `avg_position`, `ctr`
- `content_age_days`, `days_since_last_update`
- `word_count`
- `engagement_rate`, `scroll_rate` only when the analytics availability flag is genuinely TRUE in the warehouse
- `days_with_impressions`, `days_with_sessions`

### Label / proxy
For the starter mechanics only: `is_declining_proxy = trend_direction == "down"`. This is a same-window proxy and is not the ideal capstone label.

The capstone target must be a **future observed outcome**, e.g. a sustained next-window decline after a non-overlapping feature window.

### Context — never model inputs
- `content_id` / warehouse `content_hash_id`: row identity and joining
- `client_id` / `client_hash_id`: grouping and honest splitting
- report date / decision date: window construction

### Excluded
- `trend_direction` and `trend_pct` when the proxy label is used: they contain the answer.
- raw client names, URLs, queries, keywords, or titles: private / not shipped.
- any product decision score or action flag: circular decision leakage.
- future-window measurements: temporal leakage.
- availability FALSE/NULL analytics values interpreted as measurements: tracking-state leakage.

In [2]:
feature_fields = [
    "impressions_90d","clicks_90d","sessions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update","word_count",
    "engagement_rate","scroll_rate","days_with_impressions","days_with_sessions"
]
label_sources = ["trend_direction","trend_pct"]
context_fields = ["content_id","client_id"]

contract = {
    "features": feature_fields,
    "label_sources_for_starter_proxy": label_sources,
    "context": context_fields,
    "excluded": [
        "trend_direction/trend_pct as features",
        "raw identifying fields",
        "product decision flags",
        "future-window measurements"
    ]
}
print(json.dumps(contract, indent=2))


{
  "features": [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "engagement_rate",
    "scroll_rate",
    "days_with_impressions",
    "days_with_sessions"
  ],
  "label_sources_for_starter_proxy": [
    "trend_direction",
    "trend_pct"
  ],
  "context": [
    "content_id",
    "client_id"
  ],
  "excluded": [
    "trend_direction/trend_pct as features",
    "raw identifying fields",
    "product decision flags",
    "future-window measurements"
  ]
}


## 3. Verify it with queries (grain, counts, missing values, windows)

The executable local checks verify the starter grain, row/client counts, target base rate, and patterned missingness. The remote block then defines the exact checks required for the gated warehouse mid-panel month:

1. `COUNT(*)`, `MIN(report_date)`, `MAX(report_date)` for March 2026.
2. Grain probe on `report_date, client_hash_id, content_hash_id` — it must return zero duplicates.
3. Analytics availability checked with **`IS TRUE`**, never by treating zero-filled or NULL rows as valid engagement.

If `HF_TOKEN` is absent, the notebook reports that the warehouse query receipt is still pending rather than pretending it ran.

In [3]:
import numpy as np

df["is_declining_proxy"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("=== Starter verification ===")
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Proxy positives:", int(df["is_declining_proxy"].sum()))
print("Proxy rate:", round(df["is_declining_proxy"].mean(), 3))

missing = (
    df[feature_fields]
      .isna()
      .mean()
      .sort_values(ascending=False)
      .head(8)
)
print("\nTop feature missingness rates:")
print(missing.round(3).to_string())

assert df["content_id"].is_unique
assert set(label_sources).isdisjoint(feature_fields)

print("\n=== Warehouse release contract ===")
release_facts = {
    "dim_clients_rows": 104,
    "dim_content_rows": 519606,
    "daily_fact_rows": 78835655,
    "daily_min_date": "2025-01-27",
    "daily_max_date": "2026-06-30",
    "development_partition": "month=2026-03",
    "sealed_final_month": "2026-06"
}
print(json.dumps(release_facts, indent=2))

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])
    rel = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

    month_stats = con.sql(f"""
        SELECT
          COUNT(*) AS rows,
          MIN(report_date) AS min_date,
          MAX(report_date) AS max_date,
          COUNT(DISTINCT client_hash_id) AS clients,
          COUNT(DISTINCT content_hash_id) AS content_items,
          AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_true_share
        FROM {rel}
    """).df()
    print("\nMarch 2026 warehouse stats:")
    print(month_stats.to_string(index=False))

    grain_dupes = con.sql(f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {rel}
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
        LIMIT 5
    """).df()
    print("\nGrain duplicate rows:", len(grain_dupes))
    assert grain_dupes.empty
else:
    print("\nWAREHOUSE QUERY RECEIPT PENDING: HF_TOKEN is not present in this execution environment.")
    print("The query is intentionally not faked. Add HF_TOKEN as a secret, then rerun this notebook.")


=== Starter verification ===
Rows: 30000
Unique content IDs: 30000
Unique clients: 32
Proxy positives: 16262
Proxy rate: 0.542

Top feature missingness rates:
word_count          0.257
scroll_rate         0.004
impressions_90d     0.000
clicks_90d          0.000
avg_position        0.000
sessions_90d        0.000
content_age_days    0.000
ctr                 0.000

=== Warehouse release contract ===
{
  "dim_clients_rows": 104,
  "dim_content_rows": 519606,
  "daily_fact_rows": 78835655,
  "daily_min_date": "2025-01-27",
  "daily_max_date": "2026-06-30",
  "development_partition": "month=2026-03",
  "sealed_final_month": "2026-06"
}

WAREHOUSE QUERY RECEIPT PENDING: HF_TOKEN is not present in this execution environment.
The query is intentionally not faked. Add HF_TOKEN as a secret, then rerun this notebook.


## 4. Data limits

This data can support **observed patterns, predictive ranking, and decision support**. It cannot establish that refreshing a page causes recovery.

Important limits:

- The warehouse is an **unbalanced panel**: clients start tracking at different dates, so a single global window can silently compare unequal histories.
- Rows before a client's GA4 start can have GA4 values zero-filled with `ga4_data_available = FALSE`; the flag can also be NULL. Analytics features must be filtered with `IS TRUE`.
- The final month is the natural future outcome window and should stay sealed while feature/label logic is developed.
- The starter proxy is same-window and derived from `trend_direction`; it is useful for mechanics, not a final causal or future-outcome target.
- Small-volume pages can produce unstable ratios. Minimum-volume rules must be fixed before evaluation.
- Hash IDs preserve grouping but reveal no real client, URL, keyword, or query identity.
- A ranked recommendation is not an automatic edit instruction; human review remains part of the product.

In [4]:
limits = {
    "unbalanced_history": True,
    "ga4_requires_is_true": True,
    "sealed_final_month": "2026-06",
    "starter_proxy_is_same_window": True,
    "causal_claim_supported": False,
    "human_review_required": True
}
print(json.dumps(limits, indent=2))


{
  "unbalanced_history": true,
  "ga4_requires_is_true": true,
  "sealed_final_month": "2026-06",
  "starter_proxy_is_same_window": true,
  "causal_claim_supported": false,
  "human_review_required": true
}


## Self-check

- [x] Every contract section is filled with markdown thinking and executable checks
- [x] Starter grain, counts, proxy rate, and missingness are verified
- [x] Fields are classified into feature / label / context / excluded buckets
- [x] Leakage sources and availability semantics are explicit
- [x] No client names, URLs, raw queries, or credentials are published
- [ ] Mid-panel warehouse query receipt executed with `HF_TOKEN`
- [ ] Submit the public repository URL on the ML-04 card